[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

Implement a **post-training quantized linear layer** using INT8 weights.

### Signature
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [37]:
class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()
        scale = weight.abs().max(dim=1, keepdim=True)[0] / 127.0
        self.register_buffer('scale', scale)

        q_weight = torch.clamp(torch.round(weight / self.scale), -128, 127)
        self.register_buffer('weight_int8', q_weight.to(torch.int8))
        self.bias = nn.Parameter(bias) if bias is not None else None

    def forward(self, x):
        out = x @ self.weight_int8.float().transpose(-2, -1)
        out = out * self.scale.view(-1)

        if self.bias is not None:
            out += self.bias
        return out


In [38]:
# 🧪 Debug
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('Output:', q(x).shape)
print('dtype:', q.weight_int8.dtype)
print('Max quant error:', (w - q.weight_int8.float() * q.scale).abs().max().item())

Output: torch.Size([2, 8])
dtype: torch.int8
Max quant error: 0.006759822368621826


In [39]:
# ✅ SUBMIT
from torch_judge import check
check('int8_quantization')


🧪 Testing: INT8 Quantized Linear (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Weight is int8 (1.3ms)
  ✅ [2/5] Values in [-128, 127] (0.5ms)
  ✅ [3/5] Dequantized close to original (2.4ms)
  ✅ [4/5] Forward output shape (0.7ms)
  ✅ [5/5] Weight is buffer not parameter (0.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (5.6ms total)
  Progress saved. Run status() to see your dashboard.

